# Lab 5.3 – Auffälligkeiten erkennen

**Starter.** Kopie in `project/playground/` bearbeiten. Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

**Regel:** Ein Tag ist auffällig, wenn der Tagesumsatz einer Region mehr als `schwelle` Standardabweichungen vom Mittelwert dieser Region im selben Monat abweicht. Ergebnis: Funktion `find_anomalies(df, schwelle=3)`, Diagramm und `auffaellige_tage.csv` in `ORDNER`.

In [ ]:
import pandas as pd
import plotly.express as px

EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
ORDNER = "project/playground/"     # hier landet auffaellige_tage.csv

df = pd.read_csv(EINGABE, parse_dates=["datum"])

# Tagesumsatz je Region: eine Zeile je Tag und Region
tage = df.groupby(["datum", "region"], as_index=False)["umsatz"].sum()
tage["monat"] = tage["datum"].dt.strftime("%Y-%m")
print(len(tage), "Zeilen (Tag × Region)")
tage.head()

## Schritt 1 (fertig): Mittelwert und Standardabweichung je Region und Monat

`transform` schreibt den Wert der Gruppe (Region + Monat) in jede Zeile dieser Gruppe. So steht neben jedem Tag der Ø Tagesumsatz seiner Region im selben Monat und die typische Abweichung davon.

In [ ]:
gruppe = tage.groupby(["region", "monat"])["umsatz"]
tage["mittel"] = gruppe.transform("mean")      # Ø Tagesumsatz der Region in diesem Monat
tage["streuung"] = gruppe.transform("std")     # Standardabweichung: typische Abweichung vom Ø

# Beispiel: Region West, November 2025
tage[(tage["region"] == "West") & (tage["monat"] == "2025-11")].round(0).tail(4)

## Schritt 2: Regel als Funktion (Aufgabe 1)

Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 3) füllen, dann die Prüfzelle ausführen.

In [ ]:
# >>> AUFGABE 1: Funktion find_anomalies(df, schwelle=3) schreiben – Prompt siehe Übung, Aufgabe 3

# <<< ENDE AUFGABE 1

In [ ]:
# Prüfzelle – nicht ändern
def pruefe_anomalien():
    if "find_anomalies" not in globals():
        return "nicht OK – Funktion find_anomalies fehlt noch (Aufgabe 1)."
    ergebnis = find_anomalies(tage)
    if not {"abstand", "auffaellig"} <= set(ergebnis.columns):
        return "nicht OK – Spalten abstand und auffaellig fehlen im Ergebnis."
    treffer = ergebnis[ergebnis["auffaellig"]]
    bekannt = [("2025-11-27", "West"), ("2026-02-10", "Ost")]
    fehlt = [f"{d} {r}" for d, r in bekannt if not ((treffer["datum"] == d) & (treffer["region"] == r)).any()]
    if fehlt:
        return "nicht OK – nicht gefunden: " + ", ".join(fehlt) + ". abstand = (umsatz - mittel) / streuung."
    if len(treffer) != 11:
        return f"nicht OK – {len(treffer)} statt 11 auffällige Tage bei Schwelle 3: Betrag des Abstands nehmen (abs)."
    if find_anomalies(tage, schwelle=4)["auffaellig"].sum() != 2:
        return "nicht OK – der Parameter schwelle wird nicht verwendet (Schwelle 4 muss 2 Tage liefern)."
    return "OK – 11 auffällige Tage bei Schwelle 3, darunter 27.11.2025 West und 10.02.2026 Ost"


print(pruefe_anomalien())

## Schritt 3 (fertig): Diagramm und Liste der auffälligen Tage

In [ ]:
ergebnis = find_anomalies(tage, schwelle=3)
auffaellig = ergebnis[ergebnis["auffaellig"]]

fig = px.line(ergebnis, x="datum", y="umsatz", color="region",
              title="11 auffällige Tage bei Schwelle 3 – zwei ragen weit heraus",
              labels={"datum": "Tag", "umsatz": "Tagesumsatz in €", "region": "Region"})
fig.add_scatter(x=auffaellig["datum"], y=auffaellig["umsatz"], mode="markers", name="auffällig",
                marker=dict(size=12, color="red", symbol="circle-open", line=dict(width=3)))   # rote Kreise
fig

In [ ]:
liste = auffaellig[["datum", "region", "umsatz", "mittel", "streuung", "abstand"]].round(1)
liste.to_csv(ORDNER + "auffaellige_tage.csv", index=False)
print("Gespeichert:", ORDNER + "auffaellige_tage.csv")
liste.sort_values("abstand", ascending=False)

## Erweiterung: andere Schwellen, Fehlalarme zählen

Prompt und Lösung: Übung, Erweiterung.

In [ ]:
# Erweiterung: Schwelle 2, 3 und 4 vergleichen (siehe Übung)
